# Day 85 — Dataset

This notebook covers Tabular Data Preparation for Neural Networks, Retail Demand Dataset Pipeline (`RetailDemandDataset`), Pre-Training Data Sanity Checks, Target Shape Alignment (`.reshape(-1, 1)`), and Deterministic vs. Shuffled `DataLoader` Configuration

## Validation goals
- Prepare numeric feature matrices (`torch.float32`) and continuous regression targets (`(N, 1)`) for Nigerian retail demand forecasting
- Implement `RetailDemandDataset(Dataset)` with pre-training validation checks for matching row counts and `NaN` values inside `__init__`
- Build a modular `build_dataloaders` pipeline with CSV loading and deterministic synthetic fallback generation (`800` train rows, `200` test rows, `12` features)
- Configure `train_loader` (`shuffle=True`) and `test_loader` (`shuffle=False`) with `batch_size=32` and verify batch tensor shapes and dtypes

### 1. Preparing Tabular Data for a Neural Network

Neural networks train on dense floating-point tensors and are sensitive to unscaled features or mismatched target shapes. Before wrapping data in a Dataset and DataLoader, check four details:
- Clean Train / Test Separation: Keep training and test splits separate so feature scalers or encoders fit strictly on training data—never leaking test statistics.
- Numeric Feature Matrix (float32): All input columns must be numeric (float32), with no leftover strings, NaNs, or inf values.
- Target Alignment: For regression, cast targets to float32 and align shapes with model outputs—typically (N, 1) so predictions of shape (batch_size, 1) match targets without accidental broadcasting errors.
- Shuffling Policy: Shuffle training batches (shuffle=True) each epoch; keep test/validation order fixed (shuffle=False) for consistent evaluation.

### 2. Building the Dataset & DataLoader Pipeline (`day_85_dataset.py`)

When `retail_demand_train_processed.csv` and `retail_demand_test_processed.csv` are available on disk, we can load them directly and separate `demand_units` from the feature columns. If those files are not present in the working directory yet, a deterministic fallback generator creates a matching synthetic tabular dataset with the same feature names and column layout so you can run and verify the pipeline right away.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader, Dataset


FEATURE_COLUMNS = [
    "lag_7_sales",
    "lag_14_sales",
    "unit_price_ngn",
    "promo_flag",
    "fuel_transport_cost_ngn",
    "is_holiday_season",
    "region_lagos",
    "region_kano",
    "region_abuja",
    "region_rivers",
    "category_groceries",
    "category_fmcg",
]
TARGET_COLUMN = "demand_units"


def generate_synthetic_retail_splits(
    n_train: int = 800, n_test: int = 200, seed: int = 42
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Fallback generator matching the preprocessed retail demand schema."""
    rng = np.random.default_rng(seed)
    total = n_train + n_test

    lag_7 = rng.normal(150.0, 35.0, size=total).clip(20.0, None)
    lag_14 = rng.normal(145.0, 32.0, size=total).clip(20.0, None)
    unit_price = rng.normal(2400.0, 450.0, size=total).clip(500.0, None)
    promo = rng.binomial(1, 0.25, size=total).astype(float)
    fuel_cost = rng.normal(650.0, 140.0, size=total).clip(150.0, None)
    holiday = rng.binomial(1, 0.20, size=total).astype(float)

    regions = rng.choice(["lagos", "kano", "abuja", "rivers"], size=total, p=[0.35, 0.25, 0.20, 0.20])
    categories = rng.choice(["groceries", "fmcg"], size=total, p=[0.60, 0.40])

    # Standardize continuous features for neural network stability
    def zscore(arr: np.ndarray) -> np.ndarray:
        return (arr - arr.mean()) / (arr.std() + 1e-8)

    demand = (
        40.0
        + 0.45 * lag_7
        + 0.25 * lag_14
        - 0.015 * (unit_price - 2400.0)
        + 28.0 * promo
        - 0.02 * (fuel_cost - 650.0)
        + 35.0 * holiday
        + np.where(regions == "lagos", 22.0, 0.0)
        + np.where(regions == "kano", 12.0, 0.0)
        + rng.normal(0.0, 14.0, size=total)
    ).clip(10.0, None)

    df = pd.DataFrame({
        "lag_7_sales": zscore(lag_7),
        "lag_14_sales": zscore(lag_14),
        "unit_price_ngn": zscore(unit_price),
        "promo_flag": promo,
        "fuel_transport_cost_ngn": zscore(fuel_cost),
        "is_holiday_season": holiday,
        "region_lagos": (regions == "lagos").astype(float),
        "region_kano": (regions == "kano").astype(float),
        "region_abuja": (regions == "abuja").astype(float),
        "region_rivers": (regions == "rivers").astype(float),
        "category_groceries": (categories == "groceries").astype(float),
        "category_fmcg": (categories == "fmcg").astype(float),
        TARGET_COLUMN: np.round(demand, 2),
    })

    return df.iloc[:n_train].reset_index(drop=True), df.iloc[n_train:].reset_index(drop=True)


def load_retail_dataframes(
    train_path: str = "retail_demand_train_processed.csv",
    test_path: str = "retail_demand_test_processed.csv",
) -> tuple[pd.DataFrame, pd.DataFrame, str]:
    """Loads processed CSV splits if present, otherwise uses synthetic fallback."""
    t_file = Path(train_path)
    v_file = Path(test_path)
    if t_file.exists() and v_file.exists():
        return pd.read_csv(t_file), pd.read_csv(v_file), "disk_csv"
    train_df, test_df = generate_synthetic_retail_splits()
    return train_df, test_df, "synthetic_fallback"


class RetailDemandDataset(Dataset):
    """PyTorch Dataset wrapping tabular features and continuous demand targets."""

    def __init__(self, features: np.ndarray, targets: np.ndarray):
        if features.shape[0] != targets.shape[0]:
            raise ValueError("Features and targets must have the same number of rows.")
        if np.isnan(features).any() or np.isnan(targets).any():
            raise ValueError("Input features or targets contain NaN values.")

        self.X = torch.tensor(features, dtype=torch.float32)
        self.y = torch.tensor(targets, dtype=torch.float32).reshape(-1, 1)

    def __len__(self) -> int:
        return self.X.shape[0]

    def __getitem__(self, idx: int) -> tuple[torch.Tensor, torch.Tensor]:
        return self.X[idx], self.y[idx]


def build_dataloaders(
    batch_size: int = 32,
) -> tuple[DataLoader, DataLoader, int, str]:
    train_df, test_df, source = load_retail_dataframes()

    X_train = train_df.drop(columns=[TARGET_COLUMN]).to_numpy(dtype=np.float32)
    y_train = train_df[TARGET_COLUMN].to_numpy(dtype=np.float32)

    X_test = test_df.drop(columns=[TARGET_COLUMN]).to_numpy(dtype=np.float32)
    y_test = test_df[TARGET_COLUMN].to_numpy(dtype=np.float32)

    train_ds = RetailDemandDataset(X_train, y_train)
    test_ds = RetailDemandDataset(X_test, y_test)

    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, num_workers=0)
    test_loader = DataLoader(test_ds, batch_size=batch_size, shuffle=False, num_workers=0)

    return train_loader, test_loader, X_train.shape[1], source


if __name__ == "__main__":
    train_loader, test_loader, num_features, source = build_dataloaders(batch_size=32)
    batch_x, batch_y = next(iter(train_loader))

    print("--- Day 85: Retail Demand Dataset & DataLoaders ---")
    print(f"Data source:        {source}")
    print(f"Feature count:      {num_features}")
    print(f"Train dataset rows: {len(train_loader.dataset)} across {len(train_loader)} batches")
    print(f"Test dataset rows:  {len(test_loader.dataset)} across {len(test_loader)} batches")
    print(f"First batch X:      shape={tuple(batch_x.shape)}, dtype={batch_x.dtype}")
    print(f"First batch y:      shape={tuple(batch_y.shape)}, dtype={batch_y.dtype}")

--- Day 85: Retail Demand Dataset & DataLoaders ---
Data source:        synthetic_fallback
Feature count:      12
Train dataset rows: 800 across 25 batches
Test dataset rows:  200 across 7 batches
First batch X:      shape=(32, 12), dtype=torch.float32
First batch y:      shape=(32, 1), dtype=torch.float32


### 3. Data Sanity Checks Before Training

A quick pre-training audit catches subtle bugs before an epoch even runs:
- Shape alignment: Confirm `batch_x` is (`batch_size, num_features`) and `batch_y` is (`batch_size`, `1`).
- Target scale: Notice demand_units is in raw unit counts (around tens to hundreds) rather than standardized `z-scores`. Knowing the target scale helps you judge whether an initial MSE or MAE loss is reasonable.
- No missing or non-finite values: Checking for NaNs in __init__ prevents silent nan gradients on step one.

### Guided Question:

Why is validating for NaN values and matching row counts inside RetailDemandDataset.__init__ helpful before starting a training run?

### Answer:

Validating inputs inside `RetailDemandDataset`.__init__ catches data bugs early—when they are cheap to diagnose—instead of letting them corrupt model weights or crash a job hours into training.

**1. Catching NaN Values Early**

- Prevents silent weight corruption: Unchecked NaN values in features or demand targets can propagate through forward and backward passes, turning gradients and model weights into NaNs without an obvious crash until evaluation or loss logging fails.
- Pinpoints the data source: Checking inside __init__ immediately flags which column, feature slice, or target array contains NaNs while raw data context is still clear, rather than forcing you to trace a NaN loss back through batch collation, augmentations, and model layers.
- Avoids wasted compute: On large retail demand datasets, a training run can spend significant time building dataloaders, compiling graphs, or warming up GPUs before hitting a bad batch.

**2. Verifying Matching Row Counts Across Inputs**

Prevents silent misalignment: Retail demand datasets often pair multiple arrays or tables—such as features `X`, demand targets `y`, calendar/promo indicators, or store/SKU IDs. If one array is shorter or longer than another, rows can silently misalign or clip unevenly across epochs.
- Avoids cryptic runtime errors mid-epoch: Mismatched lengths may not trigger until a specific batch index or end-of-epoch boundary is reached inside __getitem__ or the DataLoader worker loop, making stack traces harder to read and debug.
- Enforces a clear dataset contract: Checking len(X) == len(y) (and any auxiliary arrays) in __init__ guarantees that __len__ and __getitem__ operate over a consistent index space before training begins.

**Good Practice in __init__**

- Raise explicit errors with context: Throw a `ValueError` that names the offending tensor/column and reports counts or `NaN` locations so fixes are immediate.

Keep checks fast: Vectorized checks (such as `isnan().any()` or `len()` `assertions`) run in milliseconds compared to a full training epoch.

### Guided Question:

Why do we reshape `self.y` to `(-1, 1)` when building a regression dataset for a model whose final layer has `out_features=1`?

### Answer:

Reshaping self.y to `(-1, 1)` gives each sample a feature dimension of size 1 so its shape (`batch_size, 1`) matches the model's output tensor (`batch_size, 1`).

**1. Shape Alignment with Model Predictions**

Model output shape: A regression head with out_features=1 produces predictions of shape (`batch_size, 1`)—one scalar value per item in the batch.

Target shape before reshape: A 1D target array or tensor of length N yields batches of shape (`batch_size,`).

Target shape after `.reshape(-1, 1)`: Each sample becomes a 1-element row [y_i], giving batches of shape (`batch_size, 1`) that mirror model(x).

**2. Avoiding Broadcasting Bugs in Loss Functions**

Many PyTorch loss functions (such as `nn.MSELoss`, `nn.SmoothL1Loss`, or `nn.L1Loss`) operate elementwise, but mismatched ranks can still cause subtle issues:

If pred has shape (`B, 1`) and target has shape (`B,`), broadcasting turns the difference into a (`B, B`) matrix—comparing every prediction against every target in the batch and producing a bogus loss value without raising a shape error.

Keeping both pred and target as (`B, 1`) guarantees `1-to-1` elementwise pairing along both dimensions.

**3. Consistency Across Batches and Metrics**

Uniformity with multi-output regression: If you later expand from 1 target to K targets (`out_features=K`), your pipeline already expects 2D targets of shape (`batch_size, K`).

Clean stacking and evaluation: Keeping a consistent (`batch_size, 1`) rank avoids accidental rank mismatches when concatenating predictions and targets across validation batches or passing them into metric functions.

### Guided Question:

Why does test_loader use `shuffle=False` while train_loader uses `shuffle=True`?

### Answer:

`train_loader` uses `shuffle=True` so `mini-batches` vary every epoch and do not teach the model artificial order patterns, while `test_loader` uses `shuffle=False` to keep evaluation deterministic and aligned with the original test samples.

**Why `train_loader` Uses `shuffle=True`**

- Prevents order bias in gradient updates: If training rows are sorted by date, store, SKU, or demand level, unshuffled `mini-batches` contain heavily correlated samples. Gradient updates can oscillate or drift toward whatever slice appears first.
- Improves generalization: Reshuffling each epoch gives the model fresh `mini-batch `combinations, acting as a mild regularizer that reduces overfitting to sequence or grouping order.
- Avoids boundary artifacts: When the total sample count is not divisible by batch_size, shuffling changes which leftover samples form the final partial batch across epochs instead of dropping or isolating the exact same tail every time.

**Why test_loader Uses `shuffle=False`**

- Deterministic, repeatable evaluation: Running evaluation multiple times—or comparing checkpoints and hyperparameters—produces the exact same batch order and metrics when `shuffle=False`.
- Preserves index alignment: Keeping the original order makes it straightforward to match predictions back to `test_df` rows, `timestamps`, `SKUs`, or store `IDs` for error analysis, plotting, or submission files.
- No learning updates occur on test data: Because test batches do not update weights via stochastic gradient descent, randomizing batch composition provides no training benefit.